In [1]:
!pip install -q --no-deps hi-ml-multimodal
!pip install -q faiss-cpu pydicom SimpleITK open_clip_torch

## load the image encoder (BioViL-T, falling back to BiomedCLIP)

In [2]:
import torch, numpy as np, pandas as pd
from pathlib import Path
from PIL import Image

ENCODER = None
try:
    from health_multimodal.image import get_image_inference
    from health_multimodal.image.utils import ImageModelType
    engine = get_image_inference(ImageModelType.BIOVIL_T)
    engine.to("cuda")

    def embed(path):
        with torch.no_grad():
            return engine.get_projected_global_embedding(Path(path)).float().cpu().numpy()
    ENCODER = "BioViL-T"
except Exception as e:
    print("BioViL-T failed -> using BiomedCLIP. Reason:", repr(e)[:300])
    import open_clip
    mdl, _, preprocess = open_clip.create_model_and_transforms(
        "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224")
    mdl = mdl.cuda().eval()

    def embed(path):
        x = preprocess(Image.open(path).convert("RGB")).unsqueeze(0).cuda()
        with torch.no_grad():
            return mdl.encode_image(x).float().squeeze(0).cpu().numpy()
    ENCODER = "BiomedCLIP"

print("Encoder:", ENCODER, "| embedding size:", embed(
    "/kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_normalized/1_IM-0001-4001.dcm.png").shape)

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
100%|██████████| 110M/110M [00:03<00:00, 34.5MB/s]


Encoder: BioViL-T | embedding size: (128,)


## load the clean split from Hugging Face

In [3]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
REPO = "jatinnabhoya/Radiology_Report_RAG-data"
df = pd.read_csv(hf_hub_download(REPO, "iu_xray_clean.csv", repo_type="dataset", token=HF_TOKEN))
print(df["split"].value_counts())

iu_xray_clean.csv:   0%|          | 0.00/1.17M [00:00<?, ?B/s]

split
train    2239
test      640
val       320
Name: count, dtype: int64


## embed all 3,199 X-rays (about 5–10 minutes)

In [4]:
from tqdm.auto import tqdm

embs = np.stack([embed(p) for p in tqdm(df["image_path"])]).astype("float32")
embs /= np.linalg.norm(embs, axis=1, keepdims=True)      # L2-normalise so inner product = cosine
print("embeddings:", embs.shape)
np.save("/kaggle/working/embeddings_all.npy", embs)
df[["uid", "split"]].to_csv("/kaggle/working/embeddings_order.csv", index=False)

  0%|          | 0/3199 [00:00<?, ?it/s]

embeddings: (3199, 128)


## build the FAISS index on train images only

In [5]:
import faiss

train_mask = (df["split"] == "train").values
index = faiss.IndexFlatIP(embs.shape[1])
index.add(embs[train_mask])
train_uids = df.loc[train_mask, "uid"].values
print("index size:", index.ntotal)            # should be 2239

faiss.write_index(index, "/kaggle/working/faiss_train.index")
pd.DataFrame({"row": range(len(train_uids)), "uid": train_uids}).to_csv(
    "/kaggle/working/index_ids.csv", index=False)

index size: 2239


## sanity check: do neighbours make clinical sense?

In [6]:
q = df[df["split"] == "test"].sample(3, random_state=0)
for _, r in q.iterrows():
    v = embs[df.index[df["uid"] == r["uid"]][0]][None]
    sims, ids = index.search(v, 3)
    print("QUERY :", r["findings"][:200])
    for s, i in zip(sims[0], ids[0]):
        nb = df[df["uid"] == train_uids[i]].iloc[0]
        print(f"  {s:.3f} ->", nb["findings"][:150])
    print()

QUERY : Normal heart size. The lungs are clear without pneumothorax or large pleural effusion. The trachea is midline and .
  0.961 -> The heart is normal in size. The mediastinum is unremarkable. The lungs are hypoinflated but clear.
  0.956 -> The heart is normal in size and contour. The lungs are clear, without evidence of infiltrate. There is no pneumothorax or effusion. There is an obliqu
  0.955 -> The cardiomediastinal silhouette and pulmonary vasculature are within normal limits in size. The lungs are clear of focal airspace disease, pneumothor

QUERY : Cardiomediastinal silhouettes are within normal limits. Lungs are clear without focal consolidation, pneumothorax, or pleural effusion. Bony thorax is unremarkable.
  0.977 -> The heart is normal in size and contour. There is no mediastinal widening. The lungs are clear bilaterally. No large pleural effusion or pneumothorax.
  0.974 -> The cardiac and mediastinal silhouettes are unremarkable. The lungs are well expanded and clea

## save everything to Hugging Face

In [7]:
api = HfApi(token=HF_TOKEN)
for f in ["embeddings_all.npy", "embeddings_order.csv", "faiss_train.index", "index_ids.csv"]:
    api.upload_file(path_or_fileobj=f"/kaggle/working/{f}", path_in_repo=f"retrieval/{f}",
                    repo_id=REPO, repo_type="dataset")
print("Saved retrieval files to", REPO, "| encoder used:", ENCODER)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


Saved retrieval files to jatinnabhoya/Radiology_Report_RAG-data | encoder used: BioViL-T
